# **Monitoreo del Modelo e Identificación de Drift en Datos Post-Despliegue**

In [1]:
# ------------------------------
# 1. LIBRERÍAS Y CONFIGURACIÓN
# ------------------------------

import pandas as pd
import numpy as np
from scipy.stats import ks_2samp, chi2_contingency
from dotenv import load_dotenv
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Cargar variables de entorno

load_dotenv()

pth = Path(os.getenv("DATA_FOLDER"))

In [3]:
# ----------------------------------------------
# 2. CARGA DE DATOS (REFERENCIA VS PRODUCCIÓN)
# ----------------------------------------------


# 2.1. Cargar datos de entrenamiento que son los de referencia

df_referencia = pd.read_csv(pth / "X_base.csv")

# 2.2. Cargar datos de producción que son los logs que toma la API

try:

    df_produccion = pd.read_csv(pth / "logs_produccion.csv")

    print(f"Se cargaron {len(df_produccion)} registros")

except FileNotFoundError:

    print("Archivo no encontró")

Se cargaron 19 registros


In [4]:
# ----------------------------------------
# 3. LIMPIEZA DE LOS DATOS DE PRODUCCIÓN
# ----------------------------------------

if not df_produccion.empty:

    # Quitar columnas que añade la API

    columnas_agregadas = ["timestamp" , "prediccion_modelo", "prob_impago"]

    df_produccion_clean = df_produccion.drop(columns = [col for col in columnas_agregadas if col in df_produccion.columns])

    # Aseguramiento del orden de las columnas

    columnas_comunes = df_referencia.columns.intersection(df_produccion_clean.columns)

    df_referencia = df_referencia[columnas_comunes]

    df_produccion_clean = df_produccion_clean[columnas_comunes]

In [5]:
# --------------------------------------------
# 4. CÁLCULO PSI (POPULATION STABILITY INDEX)
# --------------------------------------------

# PSI < 0.1 : No hya cambios significativos
# 0.1 <= PSI < 0.2 : Cambio moderado
# PSI >= 0.2 : Cambio severo --> reentrenar modelo


def calcular_psi(esperado, actual, cuantiles = 10):

    # Si la variable es numérica se crean deciles/buckets

    if pd.api.types.is_numeric_dtype(esperado):

        bins = pd.qcut(esperado, q = cuantiles, retbins = True, duplicates = 'drop')[1]

        bins[0] = -np.inf
        bins[-1] = np.inf
        
        porc_esperados = pd.cut(esperado, bins = bins).value_counts(normalize = True).sort_index()
        porc_actuales = pd.cut(actual, bins = bins).value_counts(normalize = True).sort_index()
    
    else:

        # Si es categórica --> ponerlas con value_counts

        porc_esperados = esperado.value_counts(normalize = True)
        porc_actuales = actual.value_counts(normalize = True)
        
        # Alinear losd indices

        todas_categorias = set(porc_esperados.index).union(set(porc_actuales.index))
        porc_esperados = porc_esperados.reindex(todas_categorias, fill_value = 0.0001)
        porc_actuales = porc_actuales.reindex(todas_categorias, fill_value = 0.0001)

    # Evitar divisiones por cero o logaritmos de cero

    porc_esperados = np.where(porc_esperados == 0, 0.0001, porc_esperados)
    porc_actuales = np.where(porc_actuales == 0, 0.0001, porc_actuales)

    # Fórmula del PSI: Sum( (Actual% - Esperado%) * ln(Actual% / Esperado%) )

    psi_values = (porc_actuales - porc_esperados) * np.log(porc_actuales / porc_esperados)
    
    return np.sum(psi_values)

In [6]:
# --------------------------------------------------
# 5. REPORTE DATA DRIFT (KS TEST, CHI-SQUARE Y PSI)
# --------------------------------------------------


# Revisar si hay más regsitros

if df_produccion.empty or len(df_produccion) < 10:

    print("No hay suficientes registros")

else: 

    print(f"\n{'-'*60}")
    print("REPORTE DE MONITOREO DE DATA DRIFT")
    print(f"{'-'*60}")

    for col in df_referencia.columns:

        print(f"\nVariable: {col}")

        # Extraer los datos no nulos para los test

        data_referencia = df_referencia[col].dropna()
        data_produccion = df_produccion_clean[col].dropna()

        # Ejecutar PSI, sino devolver 0.0

        try:

            psi_val = calcular_psi(data_referencia, data_produccion)

        except Exception as e:

            psi_val = 0.0

            print(f"No se calcula PSI para {col}: {e}")

        # Categórica o Numérica

        if pd.api.types.is_numeric_dtype(data_referencia) and data_referencia.nunique() > 10:

            # Test de Kolmogorov-Smirnov para Numéricas

            stat, p_value = ks_2samp(data_referencia, data_produccion)

            nombre_test = "KS-Test"

        else:

            # Test de Chi-Square para Categóricas

            ref_conteos = data_referencia.value_counts()
            prod_conteos = data_produccion.value_counts()

            # Alinear series y poner 0 donde no haya categorías cruzadas

            cat_completas = list(set(ref_conteos.index) | set(prod_conteos.index))
            ref_conteos = ref_conteos.reindex(cat_completas, fill_value = 0)
            prod_conteos = prod_conteos.reindex(cat_completas, fill_value = 0)

            # Tablas de contingencia

            tabla_contingencia = [ref_conteos.values, prod_conteos.values]

            stat, p_value, _, _ = chi2_contingency(tabla_contingencia)

            nombre_test = "Chi-Squared"
        
        
        # Diagnóstico

        # 1. Diagnóstico Estadístico (p-value < 0.05 indica que las distribuciones SON diferentes)

        if p_value < 0.05:

            diag_stat = f"Drift Detectado (p-value: {p_value:.4f})"

        else:

            diag_stat = f"Estable (p-value: {p_value:.4f})"

        
        # 2. Diagnóstico PSI

        if psi_val < 0.1:

            diag_psi = f"Sin impacto (PSI: {psi_val:.4f})"

        elif psi_val < 0.2:

            diag_psi = f"Precaución (PSI: {psi_val:.4f})"

        else:

            diag_psi = f"Peligro (PSI: {psi_val:.4f})"
            
        print(f"   • {nombre_test:12}: {diag_stat}")
        print(f"   • PSI Score   : {diag_psi}")


------------------------------------------------------------
REPORTE DE MONITOREO DE DATA DRIFT
------------------------------------------------------------

Variable: tipo_credito
   • Chi-Squared : Drift Detectado (p-value: 0.0001)
   • PSI Score   : Sin impacto (PSI: 0.0668)

Variable: capital_prestado
   • KS-Test     : Drift Detectado (p-value: 0.0000)
   • PSI Score   : Peligro (PSI: 5.8128)

Variable: plazo_meses
   • KS-Test     : Drift Detectado (p-value: 0.0000)
   • PSI Score   : Peligro (PSI: 3.3592)

Variable: edad_cliente
   • KS-Test     : Estable (p-value: 0.5692)
   • PSI Score   : Peligro (PSI: 0.7999)

Variable: tipo_laboral
   • Chi-Squared : Estable (p-value: 1.0000)
   • PSI Score   : Sin impacto (PSI: 0.0004)

Variable: salario_cliente
   • KS-Test     : Drift Detectado (p-value: 0.0014)
   • PSI Score   : Peligro (PSI: 2.9244)

Variable: total_otros_prestamos
   • KS-Test     : Drift Detectado (p-value: 0.0002)
   • PSI Score   : Peligro (PSI: 4.4885)

Variable